# EA: compute, compare, and bring your own data

This notebook computes EA through the public API. EA is the arithmetic mean
of the minimum relative-acceleration magnitudes for CV–CV, CV–CTRV, CTRV–CV,
and CTRV–CTRV. The examples use the default numerical settings.

You will recompute two synthetic equal-TTC examples, adapt a custom tracking
table with no yaw rate, inspect the causal estimates, and adjust an interaction
with manual controls. All examples here are synthetic, not paper experiment
data.

The figures use Fig. 1's muted green, gold, coral, and warm background with
regular Arial. Final figures require actual Arial; an explicitly selected
fallback is for a labelled preview only.


## 1. Environment and typography

Local setup, from the checkout root:

```bash
python -m pip install -e ".[notebook]"
python -m evasive_acceleration.native_build
python -m notebook notebooks/ea_quickstart.ipynb
```

In Colab, set ALLOW_PUBLIC_BOOTSTRAP=True to allow cloning and installing
the public repository. The setup explicitly builds the native component
before computing EA and requires a C++17 g++ compiler. Do not put access
tokens into the notebook. Set SOURCE_REVISION to a branch, tag or commit
when pinning a reproduced result.

On a machine without Arial, set EA_ARIAL_FONT_PATH to your own regular Arial TTF.
Alternatively, explicitly set ALLOW_FONT_FALLBACK=True below for a preview;
its substituted family will be printed and recorded. No font is downloaded.


In [ ]:
import os
import sys
import subprocess
import tempfile
from pathlib import Path

SOURCE_REVISION = "main"  # Branch/tag/commit in the official EA repository.
ALLOW_PUBLIC_BOOTSTRAP = False  # Explicit opt-in to clone/install in Colab.
ALLOW_FONT_FALLBACK = os.environ.get("EA_ALLOW_FONT_FALLBACK", "") == "1"

def find_checkout():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "examples/comparison_case.py").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    return None

ROOT = find_checkout()
if ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Open this notebook inside the source checkout; see the local setup above.")
    if not ALLOW_PUBLIC_BOOTSTRAP:
        raise RuntimeError(
            "Enable ALLOW_PUBLIC_BOOTSTRAP to clone and install the public EA repository "
            "in this Colab session, or run inside a prepared source checkout. "
            "Never put a GitHub token in this notebook."
        )
    ROOT = Path(tempfile.mkdtemp(prefix="ea-source-")) / "evasive-acceleration"
    subprocess.run([
        "git", "clone", "--no-checkout", "--depth", "1",
        "https://github.com/AutoChengh/evasive-acceleration.git", str(ROOT),
    ], check=True)
    if SOURCE_REVISION.startswith("-"):
        raise ValueError("SOURCE_REVISION must be a branch, tag, or commit, not a Git option.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", SOURCE_REVISION], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    if not (ROOT / "examples/comparison_case.py").is_file():
        raise RuntimeError("SOURCE_REVISION does not include the required EA examples; choose a published revision.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebook]"], check=True)
    subprocess.run([sys.executable, "-m", "evasive_acceleration.native_build"], cwd=ROOT, check=True)

os.environ["EA_ALLOW_FONT_FALLBACK"] = "1" if ALLOW_FONT_FALLBACK else "0"
sys.path.insert(0, str(ROOT / "examples"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import evasive_acceleration as ea_package
import core_ea as ea_core
import baseline_risk_metrics as ea_baselines

for module, expected in (
    (ea_package, ROOT / "src/evasive_acceleration/__init__.py"),
    (ea_core, ROOT / "src/core_ea.py"),
    (ea_baselines, ROOT / "src/baseline_risk_metrics.py"),
):
    if Path(module.__file__).resolve() != expected.resolve():
        raise RuntimeError(
            "This kernel imported a different EA checkout. Install this ROOT with "
            + sys.executable + ' -m pip install -e ".[notebook]", then restart the kernel.'
        )

from evasive_acceleration import RoadUserState, compute_ea
from evasive_acceleration.plotting import (
    FIG1_COLORS, figure_style, resolve_figure_font, plot_pair_state, plot_risk_history,
)
from baseline_risk_metrics import compute_real_time_metrics_dict
from comparison_case import run_comparison, plot_comparison
from from_dataframe import to_pair_dataframe, compute_from_dataframe

get_ipython().run_line_magic("matplotlib", "inline")
FONT = resolve_figure_font()
display(HTML(
    "<style>"
    ".text_cell_render, .jp-MarkdownOutput, .dataframe, .widget-label {"
    "font-family:'" + FONT["actual_family"] + "',sans-serif !important; color:#404040;}"
    ".text_cell_render h1,.text_cell_render h2,.text_cell_render h3,"
    ".jp-MarkdownOutput h1,.jp-MarkdownOutput h2,.jp-MarkdownOutput h3,"
    ".dataframe th,.widget-label {font-weight:400 !important;}"
    "</style>"
))

def show_table(frame):
    display(frame.style.set_properties(**{
        "font-family": FONT["actual_family"], "font-weight": "normal", "color": FIG1_COLORS["ink"],
    }).set_table_styles([{
        "selector": "th", "props": [
            ("font-family", FONT["actual_family"]), ("font-weight", "normal"),
            ("background-color", FIG1_COLORS["background"]),
        ],
    }]))

runs = ROOT / "outputs/notebooks/runs"
runs.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=str(runs)))
print("Font:", FONT["actual_family"], "| face:", FONT["face_style"], "| substituted:", FONT["fallback_used"])
print("Fresh output directory:", RUN_DIR)


## 2. Start with two current states

State positions are box centers in a common Cartesian frame. Headings and yaw
rates are radians and radians/second. The boxes and arrows below show current
geometry and velocity directions, not future truth or an executable EA command.


In [ ]:
a = RoadUserState(x=0, y=0, speed=10, heading=0, length=4.5, width=1.8, yaw_rate=0)
b = RoadUserState(x=20, y=0, speed=8, heading=np.pi, length=4.7, width=1.9, yaw_rate=0)
ea_value = compute_ea(a, b)
baseline = compute_real_time_metrics_dict(*a.as_tuple()[:6], *b.as_tuple()[:6])
print(f"EA = {ea_value:.6f} m/s^2 | TTC = {baseline['TTC']:.6f} s")
figure = plot_pair_state(a, b, ea_value=ea_value, ttc_value=baseline["TTC"])
display(figure)
plt.close(figure)


## 3. Equal TTC does not require equal EA

Two independent, straight-following synthetic states have TTC = 2 seconds.
Their bumper clearances and closing speeds differ. Recompute the values below
rather than relying on a hand-drawn comparison.

EA allows two-dimensional relative avoidance; it is not restricted to braking
inside a lane. A lower EA than longitudinal DRAC therefore does not by itself
prove a road-feasible control action. Separate plot panels retain metric units.


In [ ]:
comparison = run_comparison(RUN_DIR / "comparison", make_plot=True)
show_table(comparison[["case_id", "clearance_m", "closing_speed_m_s", "TTC_s", "EA_m_s2", "DRAC_m_s2"]])
assert np.allclose(comparison["TTC_s"], [2, 2])
figure = plot_comparison(comparison)
display(figure)
plt.close(figure)


## 4. Bring your own tracking table

The bundled custom table deliberately uses different column names, milliseconds,
and degrees, and contains no yaw-rate column. Replace its path with your own
table and edit the explicit mapping.

The adapter pairs exact observed timestamps only: no interpolation, sorting,
future filling, or inference of body heading from future positions. It rejects
duplicate/out-of-order timestamps. Positions/dimensions are meters and velocity
components are m/s. EA inputs specify forward speed along body
heading: reversing and velocity-heading disagreement above the configured
tolerance are rejected, not silently converted. The default tolerance is 5 degrees.

At unmatched timestamps the observations are not used by this paired example;
for yaw-rate estimation, only retained common history is available.


In [ ]:
tracks = pd.read_csv(ROOT / "examples/data/custom_tracks.csv")
COLUMNS = {
    "timestamp": "time_ms", "track_id": "object_key",
    "x": "east_m", "y": "north_m", "vx": "vx_mps", "vy": "vy_mps",
    "heading": "yaw_deg", "length": "box_length_m", "width": "box_width_m",
}
pair = to_pair_dataframe(
    tracks, actor_a="ego", actor_b="other", columns=COLUMNS,
    time_unit="ms", angle_unit="deg", y_down=False,
)
show_table(tracks.head(6))
show_table(pair.head(3))


In [ ]:
statistics = compute_from_dataframe(
    tracks, actor_a="ego", actor_b="other", output_dir=RUN_DIR / "custom",
    columns=COLUMNS, time_unit="ms", angle_unit="deg", yaw_rate_source="auto",
)
assert statistics["computed_success_frames"] == len(pair)
assert statistics["computation_failed_frames"] == 0
assert statistics["baseline_failed_frames"] == 0
computed = pd.read_csv(statistics["output_file"])
figure = plot_risk_history(computed, output_path=RUN_DIR / "custom/risk_history.png")
display(figure)
plt.close(figure)


## 5. Inspect yaw-rate provenance, not just the risk curve

No future heading is used. The first estimated observation is an uninitialized
zero placeholder, not a measured zero turn rate. Later rows use wrapped
backward differences and causal EWMA. These straight synthetic tracks then
have a valid estimated yaw rate of zero.

For measured yaw-rate input, use the provided policy. Auto preserves finite
supplied rates and estimates missing values. These policies prepare inputs
for the same EA algorithm.


In [ ]:
diagnostics = [
    "Time (s)", "EA", "EA_status",
    "EA_Yawrate_A_used_rad_s", "EA_Yawrate_A_source", "EA_Yawrate_A_initialized",
    "EA_Yawrate_B_used_rad_s", "EA_Yawrate_B_source", "EA_Yawrate_B_initialized",
]
show_table(computed[diagnostics])


## 6. Change clearance and closing speed

The leader travels at 10 m/s. The follower travels at 10 m/s plus the chosen
closing speed. Positive clearance keeps the initial bodies separated.
Press Recompute EA after changing the controls. The default call below also
runs in headless validation, independently of the widget frontend.


In [ ]:
from ipywidgets import FloatSlider, interact_manual

def inspect_following(clearance_m=4.0, closing_speed_m_s=2.0):
    follower = RoadUserState(0, 0, 10 + closing_speed_m_s, 0, 4.5, 1.8, 0)
    leader = RoadUserState(4.5 + clearance_m, 0, 10, 0, 4.5, 1.8, 0)
    value = compute_ea(follower, leader)
    metrics = compute_real_time_metrics_dict(*follower.as_tuple()[:6], *leader.as_tuple()[:6])
    figure = plot_pair_state(follower, leader, ea_value=value, ttc_value=metrics["TTC"])
    display(figure)
    plt.close(figure)
    print(f"EA = {value:.6f} m/s^2; TTC = {metrics['TTC']:.6f} s. No risk class is assigned.")

inspect_following()
controls = interact_manual.options(manual_name="Recompute EA")(
    inspect_following,
    clearance_m=FloatSlider(value=4, min=2, max=35, step=1, description="Clearance (m)", continuous_update=False),
    closing_speed_m_s=FloatSlider(value=2, min=1, max=12, step=0.5, description="Closing (m/s)", continuous_update=False),
)


## 7. Save a reproducible record

The comparison CSV records full states; its JSON records default settings,
font provenance, and an input SHA-256. The batch JSON records solver and yaw-rate settings. The
summary below adds the source commit, dirty-tree status, helper/source-file
hashes, adapter settings, and environment versions. A fresh directory keeps notebook reruns separate.


In [ ]:
import hashlib
import importlib.metadata
import json

revision = subprocess.run(
    ["git", "-c", "safe.directory=" + ROOT.as_posix(), "-C", str(ROOT), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=False,
)
dirty = subprocess.run(
    ["git", "-c", "safe.directory=" + ROOT.as_posix(), "-C", str(ROOT), "status", "--porcelain"],
    capture_output=True, text=True, check=False,
)
versions = {}
for package in ("evasive-acceleration", "numpy", "pandas", "matplotlib", "ipywidgets", "numba"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not installed"
summary = {
    "synthetic": True, "source_commit": revision.stdout.strip() if revision.returncode == 0 else None,
    "source_tree_dirty": bool(dirty.stdout.strip()) if dirty.returncode == 0 else None,
    "font": FONT, "palette": FIG1_COLORS, "versions": versions,
    "core_id": ea_core.CORE_ID, "core_provenance": ea_core.get_core_provenance(),
    "adapter_settings": statistics["adapter"],
    "source_sha256": {
        name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
        for name in (
            "src/core_ea.py", "src/baseline_risk_metrics.py",
            "src/evasive_acceleration/runtime.py",
            "src/evasive_acceleration/native_build.py", "src/yaw_rate.py",
            "src/evasive_acceleration/plotting.py", "examples/comparison_case.py",
            "examples/from_dataframe.py", "notebooks/ea_quickstart.ipynb",
        )
    },
    "custom_input_sha256": hashlib.sha256((ROOT / "examples/data/custom_tracks.csv").read_bytes()).hexdigest(),
    "custom_success_frames": statistics["computed_success_frames"],
    "interpretation": "EA is conditional on nominal motion assumptions, not collision probability or a control command.",
}
(RUN_DIR / "notebook_run.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Artifacts saved to:", RUN_DIR)
print("Actual font:", FONT["actual_family"], "| fallback used:", FONT["fallback_used"])
print("Source commit:", summary["source_commit"])


## Next steps

- Use the complete CLI examples and data-adaptation guide for your own dataset.
- For auditing historical paper results, follow [the archive guide](../docs/history.md)
  and preserve its original input yaw rates. Use causal EWMA for your own
  data when yaw rate is missing.
- EA values are in m/s². Zero means no added acceleration was needed over the
  evaluated futures; current contact or no feasible searched acceleration gives
  infinity. Missing or invalid computations remain unavailable, not zero.
- Baselines retain their own special-value rules; inspect the output diagnostics.
- Record inputs, numerical settings, software versions, and the source revision
  with any exported figure or result table.
